## One-time setup

Run these two cells once per Colab runtime. You don't need to re-run them just to pick up new commits or restart the server — that's what the "Run server" cell below is for.

### 1. Environment + packages (micromamba, PyTorch, Chatterbox)

In [ ]:
%%bash
set -euo pipefail

cd /content
MICROMAMBA="/content/bin/micromamba"

ts() { date +"[%Y-%m-%d %H:%M:%S]"; }

# --- Create isolated Python 3.11 environment (skips if it already exists) ---
if [ ! -x "$MICROMAMBA" ]; then
    echo "$(ts) Downloading micromamba..."
    curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
fi
if ! "$MICROMAMBA" env list | grep -q "cb311"; then
    echo "$(ts) Creating cb311 environment..."
    "$MICROMAMBA" create -y -n cb311 -c conda-forge python=3.11 pip
else
    echo "$(ts) cb311 environment already exists, skipping creation."
fi

# --- Install PyTorch (CUDA 12.1) + Chatterbox (Turbo) ---
echo "$(ts) Upgrading pip tooling inside cb311..."
"$MICROMAMBA" run -n cb311 python -m pip install -U pip setuptools wheel --progress-bar on

echo "$(ts) Installing PyTorch 2.5.1 (CUDA 12.1)... (this can take a while)"
"$MICROMAMBA" run -n cb311 pip install \
  --progress-bar on \
  torch==2.5.1+cu121 torchaudio==2.5.1+cu121 torchvision==0.20.1+cu121 \
  --index-url https://download.pytorch.org/whl/cu121

echo "$(ts) Installing Chatterbox package (from GitHub, no-cache, upgrade)..."
"$MICROMAMBA" run -n cb311 pip uninstall -y chatterbox-tts chatterbox || true
"$MICROMAMBA" run -n cb311 pip install \
  --no-cache-dir --upgrade \
  --progress-bar on \
  "chatterbox-tts @ git+https://github.com/devnen/chatterbox-v2.git@master"

echo "$(ts) Installing s3tokenizer + onnx (--no-deps to avoid protobuf conflict)..."
"$MICROMAMBA" run -n cb311 pip install --no-deps s3tokenizer==0.3.0 onnx==1.16.0

echo "$(ts) Force-upgrading protobuf for onnx compatibility..."
"$MICROMAMBA" run -n cb311 pip install --no-deps --force-reinstall "protobuf>=4.25.0"

echo "$(ts) ✅ Installation complete!"

# --- Verify GPU + verify Turbo won't require Hugging Face tokens ---
echo "$(ts) Verifying install..."
"$MICROMAMBA" run -n cb311 python - <<'PY'
import inspect, torch
import chatterbox.tts_turbo as t

print("✅ torch:", torch.__version__)
print("✅ cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("✅ gpu:", torch.cuda.get_device_name(0))

src = inspect.getsource(t.ChatterboxTurboTTS.from_pretrained)

# Heuristic check for the common buggy pattern that forces token=True semantics
markers = [" or True", "token=True", "token = True", "use_auth_token=True"]
hits = [m for m in markers if m in src]
print("Heuristic auth-forcing markers found:", hits)

if hits:
    raise SystemExit(
        "\n❌ This install still appears to force HF auth. Re-run this cell.\n"
    )

print("✅ Looks good: Turbo should download without requiring user tokens.")
PY


### 2. Drive, clone your fork, server deps, voice + config

In [ ]:
# @title 2. One-time setup: Drive, clone repo, server deps, config
import os, shutil, subprocess
from pathlib import Path
import yaml
from google.colab import userdata, drive

# ==== EDIT THESE IF YOU WANT DIFFERENT DEFAULTS ====
PORT = 8004
REPO_OWNER = "michtai"
REPO_NAME = "chatterbox"
REPO_DIR = Path(f"/content/{REPO_NAME}")
DRIVE_ROOT = Path("/content/drive/MyDrive/chatterbox")   # everything lives under here
DRIVE_OUTPUTS_DIR = DRIVE_ROOT / "outputs"
VOICE_FILENAME = "delightful_really_soft.wav"            # committed under voices/ in your fork
CHUNK_SIZE = 130  # lower than the 240 default — big chunks were cramming multiple
                  # short dialogue turns into one TTS call, causing static/dropped/
                  # garbled audio at chunk boundaries
GENERATION_DEFAULTS = {
    "temperature": 0.65,
    "exaggeration": 0.4,
    "cfg_weight": 0.5,
    "seed": 1818,
    "speed_factor": 1.0,
}
# =====================================================

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")  # Colab secret, needed if the fork is private
CLONE_URL = f"https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
LOG_STDOUT = "/content/chatterbox_server_stdout.log"

def sh(cmd, check=False, cwd=None):
    return subprocess.run(["bash", "-lc", cmd], check=check, cwd=cwd)

# === Mount Google Drive so generated output persists across sessions ===
print("=== Mounting Google Drive ===")
drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"✅ Generated audio will be saved to: {DRIVE_OUTPUTS_DIR}")

# === Clone your fork (the "run server" cell does `git pull` on this same
# clone, so it's only cloned fresh here, once per runtime) ===
print("\n=== Cloning your fork ===")
sh(f"rm -rf {REPO_DIR}", check=False)
sh(f"git clone {CLONE_URL} {REPO_DIR}", check=True)

print("\n=== Quick system checks ===")
sh("nvidia-smi || true", check=False, cwd=REPO_DIR)

print("\n=== Installing server requirements ===")
if (REPO_DIR / "requirements-nvidia.txt").exists():
    sh("/content/bin/micromamba run -n cb311 pip install -r requirements-nvidia.txt", check=False, cwd=REPO_DIR)
else:
    sh(
        "/content/bin/micromamba run -n cb311 pip install "
        "fastapi 'uvicorn[standard]' pyyaml soundfile librosa safetensors "
        "python-multipart requests jinja2 watchdog aiofiles unidecode inflect tqdm "
        "pydub audiotsm praat-parselmouth",
        check=False, cwd=REPO_DIR
    )
sh("/content/bin/micromamba run -n cb311 pip install --no-deps --force-reinstall 'protobuf>=4.25.0'", check=False)

# === Patch chatterbox to make Perth watermarker gracefully optional ===
# Perth can fail to initialize on some environments; without this patch
# the server crashes with "'NoneType' object is not callable".
# Same patch that start.py applies via _patch_chatterbox_watermarker().
print("\n=== Applying watermarker patch ===")
SITE_PKG = "/root/.local/share/mamba/envs/cb311/lib/python3.11/site-packages"
CB_DIR = Path(SITE_PKG) / "chatterbox"
SENTINEL = "# [patched: watermarker made optional]"
TARGET = "self.watermarker = perth.PerthImplicitWatermarker()"
patched = 0
for fname in ["tts.py", "tts_turbo.py", "mtl_tts.py", "vc.py"]:
    fp = CB_DIR / fname
    if not fp.exists():
        continue
    content = fp.read_text(encoding="utf-8")
    if SENTINEL in content or TARGET not in content:
        continue
    lines = content.split("\n")
    new_lines = []
    for line in lines:
        if TARGET in line and line.lstrip().startswith("self."):
            ind = line[:len(line) - len(line.lstrip())]
            new_lines.append(f"{ind}{SENTINEL}")
            new_lines.append(f"{ind}try:")
            new_lines.append(f"{ind}    self.watermarker = perth.PerthImplicitWatermarker()")
            new_lines.append(f"{ind}except Exception:")
            new_lines.append(f"{ind}    class _NoOpWatermarker:")
            new_lines.append(f"{ind}        def apply_watermark(self, wav, *args, **kwargs):")
            new_lines.append(f"{ind}            return wav")
            new_lines.append(f"{ind}    self.watermarker = _NoOpWatermarker()")
        else:
            new_lines.append(line)
    fp.write_text("\n".join(new_lines), encoding="utf-8")
    print(f"  Patched {fname}")
    patched += 1
print(f"  {patched} file(s) patched for optional watermarking" if patched else "  No patching needed")


def apply_voice_and_config():
    """
    Copies the committed voice into reference_audio/ (if needed) and writes
    config.yaml with the active model, voice, output dir, and generation
    defaults. Called here for the initial setup, and again by the "run
    server" cell after every `git pull`, since config.yaml is a tracked
    file and a pull could otherwise overwrite these local settings.
    """
    voices_dir = REPO_DIR / "voices"
    reference_dir = REPO_DIR / "reference_audio"
    reference_dir.mkdir(parents=True, exist_ok=True)

    voice_in_voices_dir = voices_dir / VOICE_FILENAME
    if not voice_in_voices_dir.exists():
        raise SystemExit(
            f"Expected {voice_in_voices_dir} in the cloned repo but it's missing. "
            f"Check that {VOICE_FILENAME} is actually committed under voices/ in your fork."
        )
    if not (reference_dir / VOICE_FILENAME).exists():
        shutil.copy(voice_in_voices_dir, reference_dir / VOICE_FILENAME)

    config_path = REPO_DIR / "config.yaml"
    with open(config_path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    cfg.setdefault("model", {})
    cfg["model"]["repo_id"] = "chatterbox"  # Default active model: Chatterbox Original (English)

    cfg.setdefault("tts_engine", {})
    cfg["tts_engine"]["default_voice_id"] = VOICE_FILENAME

    cfg.setdefault("paths", {})
    cfg["paths"]["output"] = str(DRIVE_OUTPUTS_DIR)

    cfg.setdefault("audio_output", {})
    cfg["audio_output"]["save_to_disk"] = True  # actually write generated wavs to the output folder above

    cfg.setdefault("generation_defaults", {})
    cfg["generation_defaults"].update(GENERATION_DEFAULTS)

    cfg.setdefault("ui_state", {})
    cfg["ui_state"]["last_text"] = "Type your text here."  # non-empty so the UI doesn't auto-load a preset and overwrite the sliders below
    cfg["ui_state"]["last_voice_mode"] = "predefined"
    cfg["ui_state"]["last_predefined_voice"] = VOICE_FILENAME
    cfg["ui_state"]["last_reference_file"] = VOICE_FILENAME
    cfg["ui_state"]["last_seed"] = GENERATION_DEFAULTS["seed"]
    # [patched: chunk_size actually lives under ui_state, not a top-level "chunking"
    # key — that key doesn't exist in the app's schema and was silently ignored.
    # ui_state.last_chunk_size is what the web UI's chunk-size slider reads on
    # load, and the slider's value is what gets sent as chunk_size on every
    # /tts request (falls back to a default of 240 if this isn't set).
    cfg["ui_state"]["last_chunk_size"] = CHUNK_SIZE
    cfg["ui_state"]["last_split_text_enabled"] = True

    with open(config_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, default_flow_style=False, sort_keys=False)

    print(f"  ✅ Active model: {cfg['model']['repo_id']} (Chatterbox Original / English)")
    print(f"  ✅ Output directory: {DRIVE_OUTPUTS_DIR}")
    print(f"  ✅ Default voice: {VOICE_FILENAME}")
    print(f"  ✅ Generation defaults: {GENERATION_DEFAULTS}")


print("\n=== Configuring voice + output folder + generation defaults ===")
apply_voice_and_config()

print("\n✅ One-time setup complete. Run the next cell to start the server.")


## Run the server

Re-run this cell any time: to start the server, to restart it, or to pick up new commits you've pushed to your fork (it runs `git fetch` + `git reset --hard origin/HEAD` before launching, then re-applies your voice/config settings since `config.yaml` is tracked and the pull could otherwise overwrite them).

In [ ]:
# @title 3. Run server (pulls your latest repo changes, then starts it)
import os, re, socket, subprocess
import requests
from pathlib import Path

def port_open(host="127.0.0.1", port=PORT, timeout=0.25):
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False

# === Pull any changes you've pushed to your fork since the last run ===
# `reset --hard` discards local edits (e.g. the config.yaml rewrite below)
# before pulling, so this never conflicts — config.yaml is regenerated by
# apply_voice_and_config() right after, on every run of this cell.
print("=== Pulling latest changes from your fork ===")
sh("git fetch origin", check=True, cwd=REPO_DIR)
sh("git reset --hard origin/HEAD", check=True, cwd=REPO_DIR)
sh("git log -1 --oneline", check=False, cwd=REPO_DIR)

print("\n=== Re-applying voice + config (in case the pull touched config.yaml) ===")
apply_voice_and_config()

print("\n=== Removing old stdout log ===")
Path(LOG_STDOUT).unlink(missing_ok=True)

print("\n=== Starting server with LIVE logs ===")
print("Log file:", LOG_STDOUT)
print("To stop the server, run the last cell in this notebook.\n")

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

# Put HF cache somewhere inspectable/persistent for this runtime
env["HF_HOME"] = "/content/hf_home"
env["TRANSFORMERS_CACHE"] = "/content/hf_home/transformers"
env["HF_HUB_CACHE"] = "/content/hf_home/hub"
Path(env["HF_HOME"]).mkdir(parents=True, exist_ok=True)

proc = subprocess.Popen(
    ["/content/bin/micromamba", "run", "-n", "cb311", "python", "-u", "server.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
    cwd=REPO_DIR,
)

_progress_state = {"dots": 0, "last_current": -1}

def _handle_progress_line(raw_line):
    """
    Returns True if `raw_line` is a Chatterbox/tqdm 'Sampling: NN%|...' progress
    update (and should NOT be printed as-is). Instead, prints a '.' for each
    percent of progress, wrapping to a new line every 25 dots (so at most
    4 lines of dots appear per 1000-sample generation).
    """
    m = re.search(r"Sampling:.*?(\d+)/(\d+)\s*\[", raw_line)
    if not m:
        return False

    current, total = int(m.group(1)), int(m.group(2))

    # A new bar has started (current count dropped back down) — close out
    # the previous line of dots if it wasn't already at a 25-dot boundary.
    if current < _progress_state["last_current"]:
        if _progress_state["dots"] % 25 != 0:
            print()
        _progress_state["dots"] = 0

    _progress_state["last_current"] = current
    target_dots = min(100, int(current / total * 100)) if total else 0

    while _progress_state["dots"] < target_dots:
        print(".", end="", flush=True)
        _progress_state["dots"] += 1
        if _progress_state["dots"] % 25 == 0:
            print()

    return True

with open(LOG_STDOUT, "w", encoding="utf-8", errors="replace") as f:
    shown_link = False
    while True:
        line = proc.stdout.readline()
        if line:
            if not _handle_progress_line(line):
                print(line, end="")
            f.write(line)  # full detail still goes to the log file
            f.flush()

        if (not shown_link) and port_open():
            shown_link = True
            print("\n" + "="*60)
            print("=== Server is ready! ===")
            print("="*60)
            from google.colab.output import eval_js
            proxy_url = eval_js(f'google.colab.kernel.proxyPort({PORT})')
            print(f"\n🌐 Open this URL in a new browser tab:\n\n   {proxy_url}\n")
            print(f"📚 API docs:  {proxy_url}docs\n")
            print("="*60 + "\n")
            # Verify model load status via server endpoint
            try:
                mi = requests.get(f"http://127.0.0.1:{PORT}/api/model-info", timeout=2).json()
                print("\n/api/model-info:", mi)
            except Exception as e:
                print("\n/api/model-info query failed:", repr(e))

        if proc.poll() is not None:
            print("\n=== Server process exited with code", proc.returncode, "===")
            break


## Stop the server

Run any time you want to free port 8004 — e.g. before re-running the cell above.

In [ ]:
%%bash
PORT=8004

echo "PIDs listening on port $PORT:"
sudo lsof -t -i:$PORT || true

echo "Killing..."
sudo lsof -t -i:$PORT | xargs -r sudo kill -9

echo "Verify nothing is listening:"
sudo lsof -i:$PORT || true
